# 27장 · LangGraph 조건부 간선과 재검색

조건부 간선은 상태에 따라 서로 다른 노드로 이어집니다. 빈 검색 결과를 다시 찾게 할 때는 반복을 끝낼 조건도 필요합니다.

‘책을 빌리는 기간?’을 첫 검색에서 찾지 못하면 질문을 수정하고 다시 검색합니다. 조건부 간선이 문서 목록과 attempts를 읽어 답변·재시도·종료 중 한 경로를 선택합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## add_conditional_edges()와 라우팅 함수

search가 시도 횟수를 올린 직후 route가 문서 유무와 attempts를 읽습니다. 문서가 있으면 답을 쓰고, 없고 한 번만 찾았다면 검색어를 바꿉니다. 두 번 찾고도 빈 목록이면 멈춥니다.

**조건부 간선에서 선택되는 경로**　선로가 갈라지는 지점에서 열차는 한쪽으로만 갑니다. route()의 반환값을 바꾸면 어느 노드가 실행되는지 코드와 대조하세요.

### 셀 01 · route()의 분기 반환값

검색을 한 번 마쳤고 문서를 못 찾았습니다. 다음에 갈 곳을 고르는 함수만 먼저 실행하세요.

In [1]:
def route(state):
    if state["documents"]:
        return "answer"
    if state["attempts"] < 2:
        return "retry"
    return "stop"

after_search = {'question': '책을 빌리는 기간?', 'documents': [], 'answer': '', 'attempts': 1}
print(route(after_search))

retry


retry가 출력된 줄에서는 아직 두 번째 검색을 하지 않았습니다. route는 경로 이름을 고르고, 그래프에 등록한 조건부 간선이 그 이름을 다음 노드와 연결합니다. 첫 검색에 문서가 있었을 때와 비어 있었을 때 어느 노드를 한 번 더 지나게 되는지 비교하세요.

### 조건 함수의 경로 이름

<p>search가 documents를 갱신한 뒤 route가 현재 상태를 읽습니다. route의 역할은 답변을 만드는 것이 아니라 answer·retry·stop 중 한 문자열을 반환하는 것입니다. add_conditional_edges의 대응표가 그 문자열을 등록한 노드 이름과 연결합니다.</p>

<p>직접 성공에서는 한 번 검색하고 종료합니다. 실패라고 무조건 다시 검색하면 같은 입력과 결과를 끝없이 반복할 수 있습니다. 이번 예제는 문서가 없고 attempts가 2보다 작을 때만 재시도합니다.</p>

## 질의 재작성과 재검색 횟수 제한

질문을 고치는 함수가 여러 번 불려도 검색 횟수와 같으리라는 보장은 없습니다. 실제 검색을 실행한 search에서만 attempts를 하나 올립니다. 반복을 제한하는 판단은 이 갱신이 반영된 뒤에 내립니다.

**검색 상태에 따른 다음 노드**　첫 검색 결과가 비면 재검색하고 두 번째에도 비면 종료합니다.

[웹 교재에서 열기](https://langchain-lecture.vercel.app/chapter27/index.html#practice-space)

### 셀 02 · add_conditional_edges()로 경로 등록

이미 실행한 route는 그대로 씁니다. search가 attempts를 올리는 줄과 rewrite 뒤에 다시 search로 돌아가는 엣지를 찾아보세요.

셀 01에서 나온 retry는 아직 최종 답이 아닙니다. 그래프는 rewrite로 이동한 뒤 search에 다시 들어갑니다. 다음 검색에서도 비었을 때 어떤 조건이 반복을 끝내는지 찾아보세요.

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    question: str
    documents: list[str]
    answer: str
    attempts: int

def search(state: State) -> dict:
    if not state["question"].strip():
        raise ValueError("질문을 입력하세요.")
    documents = ["도서 대출 기간은 14일입니다."] if "대출" in state["question"] else []
    return {"documents": documents, "attempts": state["attempts"] + 1}

def rewrite(state: State) -> dict:
    return {"question": state["question"].replace("책", "대출")}

def answer(state: State) -> dict:
    return {"answer": " / ".join(state["documents"])}

def fallback(state: State) -> dict:
    return {"answer": "두 번 검색했지만 근거가 없습니다. 질문을 구체화해 주세요."}

def make_graph():
    builder = StateGraph(State)
    for name, node in [("search", search), ("rewrite", rewrite),
                       ("answer_node", answer), ("fallback", fallback)]:
        builder.add_node(name, node)
    builder.add_edge(START, "search")
    builder.add_conditional_edges("search", route,
        {"answer": "answer_node", "retry": "rewrite", "stop": "fallback"})
    builder.add_edge("rewrite", "search")
    builder.add_edge("answer_node", END)
    builder.add_edge("fallback", END)
    return builder.compile()

def run(question: str) -> State:
    initial: State = {"question": question, "documents": [], "answer": "", "attempts": 0}
    return make_graph().invoke(initial, config={"recursion_limit": 20})

print(search(after_search)['attempts'])

2


같은 상태에서 검색을 한 번 더 실행해 횟수가 2가 됐습니다. 이 셀의 함수 정의를 다음 셀에서 사용합니다.

#### 두 번째 검색에서 답을 찾으면

<mark>검색 횟수가 상한에 닿았어도 문서를 찾았으면 answer 경로로 갑니다.</mark> route는 documents부터 검사합니다. ‘책을 빌리는 기간?’과 ‘주차장은?’은 둘 다 재검색할 수 있지만, 고친 질문에 검색어가 생기는지에 따라 마지막 노드가 갈립니다.

선수 개념 · 26장 LangGraph StateGraph의 상태와 노드

#### 문서 유무와 검색 횟수로 다음 노드 고르기

```mermaid
flowchart TD
s["START"] --> q["search"]
q --> r{"route · 문서가 있는가?"}
r -->|answer · 있음| a["answer_node"]
r -->|없음| b{"attempts가 2 미만?"}
b -->|retry| w["rewrite"] --> q
b -->|stop| f["fallback"]
a --> e["END"]
f --> e
```

search가 갱신한 상태를 route가 읽습니다. answer·retry·stop은 조건 함수의 반환값이고, 실제로 등록한 노드 이름은 answer_node·rewrite·fallback입니다.

### 상태에 저장하는 시도 횟수

<p>State에 attempts 정수 필드를 추가합니다. 초기 값 0에서 시작해 search가 실제 실행될 때마다 1을 더합니다. 검색 함수 실행 후 route가 읽는 값은 이미 증가한 값입니다. 첫 번째 실패 뒤에는 1이므로 재시도하고, 두 번째 실패 뒤에는 2이므로 종료합니다.</p>

<p>attempts는 이 예제에서 검색 노드 실행 횟수입니다. 전체 노드 수나 모델 토큰 수가 아닙니다. 답변 노드와 rewrite 실행을 함께 세고 싶다면 별도의 기록을 설계해야 합니다.</p>

### 재검색 순환 간선

<p>rewrite 다음에 search로 돌아가는 연결이 반복을 만듭니다. answer_node와 fallback은 END로 연결되어 더 이상 돌아가지 않습니다. 모든 경우를 반복으로 처리하지 않고 성공과 포기 경로를 분리합니다.</p>

### 한 번 고친 질문이 두 번째 검색에서 성공합니다

route의 retry는 rewrite 노드를 선택합니다. 다음 search가 비어 있지 않은 documents를 돌려주면 answer 경로로 바뀝니다.

1. search · attempts=1 = documents = []
2. rewrite · question = 대출을 빌리는 기간?
3. search · attempts=2 = documents 1건
4. answer_node · answer = 도서 대출 기간은 14일입니다.
5. END · 검색 횟수 = 2

### 셀 03 · rewrite 노드의 질의 변경

질문 한 건의 마지막 상태에서 횟수와 답을 확인하세요.

In [3]:
result = run('책을 빌리는 기간?')
print(result['attempts'], result['answer'])

2 도서 대출 기간은 14일입니다.


두 번째 검색에서 ‘대출’이 들어 있는 질문으로 문장을 찾았습니다. 첫 번째 빈 결과만 보고 바로 끝냈다면 14일이라는 답까지 가지 못했을 것입니다.

### 상태표의 세 종료 경로

<table><caption>관찰할 경로</caption><thead><tr><th scope="col">입력</th><th scope="col">노드 순서</th><th scope="col">검색 횟수</th></tr>
</thead><tbody><tr><td>대출 기간?</td><td>search → answer_node</td><td>1</td></tr>
<tr><td>책을 빌리는 기간?</td><td>search → rewrite → search → answer_node</td><td>2</td></tr>
<tr><td>주차장은?</td><td>search → rewrite → search → fallback</td><td>2</td></tr>
</tbody></table>

<p>두 번째 입력은 첫 검색에서 대출이라는 글자가 없어 실패합니다. rewrite가 책을 대출로 바꾼 뒤 다시 검색하면 근거가 생깁니다. 세 번째 입력은 rewrite로 바뀌지 않아도 횟수 상한으로 끝납니다. 질문이 달라진 사실과 답변이 좋아진 사실은 따로 확인해야 합니다.</p>

### 과제 04 · 검색 상한과 fallback

주차장 질문을 처리한 뒤 몇 번 검색했고 사용자에게 무엇을 안내하는지 출력하세요.

In [4]:
# 주차장 질문을 run에 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>result = run(&#x27;주차장은?&#x27;)
print(result[&#x27;attempts&#x27;], result[&#x27;answer&#x27;])</code></pre>
<p>실행 결과</p><pre><code>2 두 번 검색했지만 근거가 없습니다. 질문을 구체화해 주세요.</code></pre>
<p>두 번 모두 문서가 비어 fallback으로 갑니다. attempts가 2일 때도 retry로 보내면 설계한 상한을 넘게 됩니다. 프레임워크의 recursion_limit은 이런 실수를 발견하는 별도 보호 장치입니다.</p>
</details>

### 반복 상한과 실행 오류

<p>run의 recursion_limit=20은 그래프 실행 단계의 한도입니다. 검색을 두 번으로 제한하는 코드는 route의 attempts &lt; 2입니다. 정상 종료라면 주차 질문에서도 예외 대신 fallback의 안내를 받습니다.</p>

<p>공백뿐인 질문은 search가 ValueError를 내므로 route까지 도달하지 않습니다. 주차 질문은 검색 함수가 빈 목록을 정상 반환해 route에서 재시도 여부를 정합니다.</p>

<table><caption>오류 점검</caption><thead><tr><th scope="col">증상</th><th scope="col">확인</th><th scope="col">수정 방향</th></tr>
</thead><tbody><tr><td>끝나지 않거나 GraphRecursionError</td><td>attempts 증가·route 상한</td><td>검색 후 증가와 stop 연결 확인</td></tr>
<tr><td>경로 이름 오류</td><td>route 문자열·대응표·등록 이름</td><td>세 위치를 함께 비교</td></tr>
<tr><td>두 번 성공인데 실패 답변</td><td>조건 검사 순서</td><td>documents 성공 조건 우선</td></tr>
<tr><td>빈 질문 ValueError</td><td>입력 값</td><td>공백이 아닌 질문 입력</td></tr>
<tr><td>항상 문서 없음</td><td>키워드 포함 검사</td><td>가상 검색의 정확한 조건 확인</td></tr>
</tbody></table>

## 돌아보기

검색 한 번에 바로 답하는 질문도 골라 보세요. 세 질문 모두 search 다음에 route를 거칩니다. 어떤 상태에서 다음 경로가 갈라졌나요?

### 과제 05 · 재검색이 끝나는 노드 추적

주차 질문에서는 rewrite가 질문을 바꾸지 못합니다. 그래프의 stream_mode="updates"를 사용해 실제로 실행된 노드의 이름과 search가 보고한 attempts만 출력하세요.

In [ ]:
request = {"question": "주차장은?", "documents": [], "answer": "", "attempts": 0}
# 각 갱신에 들어 있는 노드 이름과 검색 횟수를 읽으세요.

<details><summary>풀이와 실행 결과 보기</summary>
<pre><code>request = {"question": "주차장은?", "documents": [], "answer": "", "attempts": 0}

for update in make_graph().stream(request, stream_mode="updates"):
    for node, change in update.items():
        if node == "search":
            print(node, change["attempts"])
        else:
            print(node)</code></pre>
<p>실행 결과</p><pre><code>search 1
rewrite
search 2
fallback</code></pre>
<p>첫 번째 빈 검색은 rewrite로 가지만 두 번째 빈 검색은 fallback으로 갑니다. route는 다음 노드를 고르는 조건 함수이므로 실행 노드 이름으로 따로 출력되지 않습니다.</p>
</details>

## 검색어가 그대로면 다시 검색하기 전에 끝내기

기존 그래프는 검색이 비면 한 번 질의를 바꿉니다. 그런데 주차 질문에는 rewrite가 바꿀 표현이 없어 똑같은 질의를 두 번 검색합니다. 재시도 횟수에 더해 “바뀐 질의가 있는가”를 판단해 봅니다.

### 셀 06 · 재조회할 질의가 실제로 달라졌는지 확인하기

주차 질문의 첫 검색 직후에 기존 rewrite를 호출합니다. 다음 검색이 받을 문자열을 이전 값과 대조하세요.

In [5]:
first_search = {**request, **search(request)}
proposal = rewrite(first_search)["question"]
print(repr(first_search["question"]), repr(proposal), route(first_search))

'주차장은?' '주차장은?' retry


두 문자열은 같지만 기존 route는 retry를 돌려줍니다. attempts가 1이고 자료가 없다는 두 가지 값만 보고 결정했기 때문입니다.

### 질의 변경과 검색 상한으로 재시도 결정하기

```mermaid
flowchart TD
 s["search"] --> found{"근거가 있는가?"}
 found -->|있음| a["answer_node"]
 found -->|없음| edit{"질의가 바뀌며 상한이 남았는가?"}
 edit -->|예| r["rewrite"]
 r --> s
 edit -->|아니요| f["finish"]
 a --> done["END"]
 f --> done
```

search 결과를 먼저 살핍니다. 자료가 없을 때에만 다음 질의와 시도 횟수를 확인합니다.

### 과제 07 · 같은 질의를 반복하지 않는 조건부 간선 만들기

찾은 자료가 있으면 answer_node, 바꿀 표현이 있고 검색 상한이 남았으면 rewrite, 나머지는 finish로 보내세요. 종료 문구에는 실제 attempts를 넣습니다. 주차 질문과 “책을 빌리는 기간?”의 시도 횟수를 확인하세요.

In [6]:
# route_once를 정의하고 search 이후에 조건부 간선을 연결하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>def route_once(state):
    if state[&quot;documents&quot;]:
        return &quot;answer_node&quot;
    changed = rewrite(state)[&quot;question&quot;] != state[&quot;question&quot;]
    return &quot;rewrite&quot; if changed and state[&quot;attempts&quot;] &lt; 2 else &quot;finish&quot;

revised = StateGraph(State)
revised.add_node(&quot;search&quot;, search)
revised.add_node(&quot;rewrite&quot;, rewrite)
revised.add_node(&quot;answer_node&quot;, answer)
revised.add_node(&quot;finish&quot;, lambda s: {&quot;answer&quot;: f&quot;{s[&#x27;attempts&#x27;]}회 검색 후 종료&quot;})
revised.add_edge(START, &quot;search&quot;)
revised.add_conditional_edges(&quot;search&quot;, route_once,
                             {n: n for n in (&quot;answer_node&quot;, &quot;rewrite&quot;, &quot;finish&quot;)})
revised.add_edge(&quot;rewrite&quot;, &quot;search&quot;)
revised.add_edge(&quot;finish&quot;, END)
revised.add_edge(&quot;answer_node&quot;, END)
smart_search = revised.compile()
print(smart_search.invoke(request)[&quot;answer&quot;])
print(smart_search.invoke({**request, &quot;question&quot;: &quot;책을 빌리는 기간?&quot;})[&quot;attempts&quot;])</code></pre>
<p>실행 결과</p><pre><code>1회 검색 후 종료
2</code></pre>
<p>주차는 1회에서 끝납니다. 책은 대출로 바꿀 수 있어서 두 번째 검색에서 문서를 얻습니다. 마지막 시도에서 자료를 찾았을 때도 답하려면 종료 횟수보다 자료 유무를 먼저 확인해야 합니다.</p>
</details>

<mark>다음 검색의 입력이 달라질 때에만 재시도가 새 근거를 찾을 기회를 얻습니다.</mark> 이 예제의 검색 함수는 같은 입력과 같은 문서에서 항상 같은 목록을 반환합니다.

나중에 모델이 재검색을 선택할 때도 어떤 입력이 달라졌는지 함께 기록하세요.